<a href="https://colab.research.google.com/github/roneshpra/PythonLearning/blob/Dev/PythonBasics/M5W5_Project_MedicalAssistant_Final_RB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [1]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
#!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q --upgrade pip
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" pip install --no-cache-dir llama-cpp-python==0.2.77 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.4/316.4 MB 119.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 6.5 MB/s eta 0:00:00


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [2]:
# For installing the libraries & downloading models from HF Hub
#!pip install huggingface_hub==0.23.2 pandas==1.5.3 tiktoken==0.6.0 pymupdf==1.25.1 langchain==0.1.1 langchain-community==0.0.13 chromadb==0.4.22 sentence-transformers==2.3.1 numpy==1.25.2 -q
!pip install huggingface_hub pandas tiktoken pymupdf langchain langchain_community chromadb sentence_transformers "numpy<2" -q

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [3]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

## Question Answering using LLM

#### Downloading and Loading the model

#### Model Loading (Mistral)

In [4]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [5]:
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


mistral-7b-instruct-v0.2.Q6_K.gguf:   0%|          | 0.00/5.94G [00:00<?, ?B/s]

In [6]:
llm = Llama(
    model_path=model_path,
    n_ctx=2300,
    n_gpu_layers=38,
    n_batch=512
)

llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from /root/.cache/huggingface/hub/models--TheBloke--Mistral-7B-Instruct-v0.2-GGUF/snapshots/3a6fbf4a41a1d52e415a4958cde6856d34b2db93/mistral-7b-instruct-v0.2.Q6_K.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loade

#### Response

In [7]:
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

In [8]:
response("What treatment options are available for managing hypertension?")


llama_print_timings:        load time =     613.46 ms
llama_print_timings:      sample time =      74.43 ms /   128 runs   (    0.58 ms per token,  1719.76 tokens per second)
llama_print_timings: prompt eval time =     613.37 ms /    12 tokens (   51.11 ms per token,    19.56 tokens per second)
llama_print_timings:        eval time =    4211.43 ms /   127 runs   (   33.16 ms per token,    30.16 tokens per second)
llama_print_timings:       total time =    4980.76 ms /   139 tokens


'\n\nHypertension, or high blood pressure, is a common condition that can increase the risk of various health problems such as heart disease, stroke, and kidney damage. The good news is that there are several effective treatment options available to help manage hypertension and reduce the risk of complications. Here are some of the most commonly used treatments:\n\n1. Lifestyle modifications: Making lifestyle changes is often the first line of defense against hypertension. This may include eating a healthy diet rich in fruits, vegetables, whole grains, and lean proteins; limiting sodium intake; getting regular physical activity'

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [13]:
user_input_1 = "What is the protocol for managing sepsis in a critical care unit?"
response(user_input_1, 200)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     613.46 ms
llama_print_timings:      sample time =     110.88 ms /   200 runs   (    0.55 ms per token,  1803.75 tokens per second)
llama_print_timings: prompt eval time =     210.75 ms /    14 tokens (   15.05 ms per token,    66.43 tokens per second)
llama_print_timings:        eval time =    6922.16 ms /   199 runs   (   34.78 ms per token,    28.75 tokens per second)
llama_print_timings:       total time =    7353.16 ms /   213 tokens


'\n\nSepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:\n\n1. Early recognition: Recognize the signs and symptoms of sepsis early and initiate treatment as soon as possible. Sepsis can present with various clinical features, including fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.\n2. ABCs: Ensure airway patency, adequate breathing, and circulatory support. Provide high-flow oxygen via a non-rebreather mask or endotracheal tube if necessary. Initiate intravenous fluids to maintain adequate blood pressure and organ perfusion.\n3. Antibiotics: Administer broad-spectrum antibiot'

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [12]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(user_input_2, 200)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     613.46 ms
llama_print_timings:      sample time =     111.36 ms /   200 runs   (    0.56 ms per token,  1795.96 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     0 tokens (    -nan ms per token,     -nan tokens per second)
llama_print_timings:        eval time =    6987.84 ms /   200 runs   (   34.94 ms per token,    28.62 tokens per second)
llama_print_timings:       total time =    7205.67 ms /   200 tokens


'\n\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small pouch-like structure that extends from the large intestine. The symptoms of appendicitis can vary from person to person, but some common signs include:\n\n1. Abdominal pain: The pain is typically located in the lower right side of the abdomen and may be constant or come and go. It may start as a mild discomfort that gradually worsens over time.\n2. Loss of appetite: People with appendicitis often lose their appetite due to abdominal pain and nausea.\n3. Nausea and vomiting: Vomiting is a common symptom of appendicitis, especially in the early stages.\n4. Fever: A fever may be present, particularly if the appendix has ruptured or perforated.\n5. Constipation or diarrhea: Some people with'

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [14]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(user_input_3, 200)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     613.46 ms
llama_print_timings:      sample time =     110.39 ms /   200 runs   (    0.55 ms per token,  1811.81 tokens per second)
llama_print_timings: prompt eval time =     230.20 ms /    36 tokens (    6.39 ms per token,   156.39 tokens per second)
llama_print_timings:        eval time =    6920.03 ms /   199 runs   (   34.77 ms per token,    28.76 tokens per second)
llama_print_timings:       total time =    7360.05 ms /   235 tokens


"\n\nSudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles. It can result in round or oval bald patches on the scalp, but it can also occur on other parts of the body such as the beard area, eyebrows, or eyelashes.\n\nThe exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Some possible triggers for this condition include stress, genetics, viral infections, and certain medications.\n\nThere are several treatments that have been shown to be effective in addressing sudden patchy hair loss:\n\n1. Corticosteroids: These are anti-inflammatory drugs that can help reduce inflammation and suppress the immune system's attack on the hair follicles. They can be applied topically or taken orally,"

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [16]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_input_4, 200)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     613.46 ms
llama_print_timings:      sample time =     114.98 ms /   200 runs   (    0.57 ms per token,  1739.42 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     0 tokens (    -nan ms per token,     -nan tokens per second)
llama_print_timings:        eval time =    7007.68 ms /   200 runs   (   35.04 ms per token,    28.54 tokens per second)
llama_print_timings:       total time =    7241.61 ms /   200 tokens


"\n\nA person who has sustained a physical injury to brain tissue, also known as a traumatic brain injury (TBI), may require various treatments depending on the severity and location of the injury. Here are some common treatments recommended for TBIs:\n\n1. Emergency care: The first priority is to ensure the person's airway is clear, they are breathing, and their heart is beating normally. In severe cases, emergency surgery may be required to remove hematomas or other obstructions.\n2. Medications: Depending on the symptoms, medications may be prescribed to manage conditions such as swelling, seizures, pain, or infections. For example, corticosteroids may be used to reduce brain swelling, and anticonvulsants may be given to prevent seizures.\n3. Rehabilitation: Rehabilitation is an essential part of the recovery process for TBI patients. Rehabilitation may include physical therapy"

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [17]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(user_input_5, 200)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     613.46 ms
llama_print_timings:      sample time =     117.30 ms /   200 runs   (    0.59 ms per token,  1704.97 tokens per second)
llama_print_timings: prompt eval time =     195.09 ms /    35 tokens (    5.57 ms per token,   179.40 tokens per second)
llama_print_timings:        eval time =    7337.95 ms /   199 runs   (   36.87 ms per token,    27.12 tokens per second)
llama_print_timings:       total time =    7767.69 ms /   234 tokens


"\n\nFirst and foremost, if you suspect that someone has fractured their leg while hiking, it's essential to ensure their safety and prevent further injury. Here are some necessary precautions:\n\n1. Keep the person calm and still: Encourage them to remain as still as possible to minimize pain and prevent worsening the injury.\n2. Assess the situation: Check for any signs of shock, such as pale skin, rapid heartbeat, or low blood pressure. If you notice these symptoms, seek medical help immediately.\n3. Immobilize the leg: Use a splint, sling, or other immobilizing device to prevent movement and reduce pain. Be sure to secure it properly without putting too much pressure on the injury.\n4. Provide first aid: Clean the wound with water and mild soap, apply an antiseptic, and cover it with a sterile dressing.\n5. Seek medical help: If"

#### Observations:

*   The above outputs from the LLM are very generic



## Question Answering using LLM with Prompt Engineering

In [18]:
system_prompt = """
You are an AI specialist.
Your task is to provide AI solutions using renowned medical manuals.
The solutions should include Diagnostic Assistance, Drug Information, Treatment Plans, Specialty Knowledge, and Critical Care Protocols.
Ensure the information is accurate, up-to-date, and based on established medical guidelines.
 """

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [23]:
user_input_11 = system_prompt+"\n"+ "What is the protocol for managing sepsis in a critical care unit?"
response(user_input_11, max_tokens=500, temperature=0.5, top_k=2)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     613.46 ms
llama_print_timings:      sample time =     300.52 ms /   500 runs   (    0.60 ms per token,  1663.78 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     0 tokens (    -nan ms per token,     -nan tokens per second)
llama_print_timings:        eval time =   18197.19 ms /   500 runs   (   36.39 ms per token,    27.48 tokens per second)
llama_print_timings:       total time =   18921.85 ms /   500 tokens


' According to the latest guidelines from the Surviving Sepsis Campaign (SSC), what are the recommended interventions for early recognition, fluid resuscitation, antibiotic therapy, and vasopressor use?\n\nAccording to the latest guidelines from the Surviving Sepsis Campaign (SSC), sepsis is defined as a life-threatening condition caused by a dysregulated host response to infection. The SSC recommends early recognition and rapid initiation of appropriate interventions to improve outcomes in sepsis patients.\n\nEarly Recognition:\nThe SSC recommends using the Sequential Organ Failure Assessment (SOFA) score or Quick Sequential [Sepsis-related] Organ Failure Assessment (qSOFA) score for early recognition of sepsis. Patients with suspected infection and an SOFA score ≥2 or qSOFA score ≥2 should be evaluated for sepsis and initiated on appropriate antibiotic therapy as soon as possible.\n\nFluid Resuscitation:\nFor fluid resuscitation, the SSC recommends using crystalloids as the initial f

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [24]:
user_input_12 = system_prompt+"\n"+ "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(user_input_12, max_tokens=500, temperature=0.5, top_k=2)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     613.46 ms
llama_print_timings:      sample time =     182.49 ms /   325 runs   (    0.56 ms per token,  1780.91 tokens per second)
llama_print_timings: prompt eval time =     228.41 ms /    32 tokens (    7.14 ms per token,   140.10 tokens per second)
llama_print_timings:        eval time =   12415.33 ms /   324 runs   (   38.32 ms per token,    26.10 tokens per second)
llama_print_timings:       total time =   13039.50 ms /   356 tokens


'\n\nAccording to the Merck Manual (merckmanuals.com), the common symptoms of appendicitis include:\n1. Sudden onset of pain, usually in the lower right abdomen, that worsens over several hours.\n2. Loss of appetite and feeling sick to your stomach (nausea).\n3. Vomiting.\n4. Fever, often high-grade.\n5. Abdominal swelling and rigidity.\n6. Pain in the lower right quadrant when walking or making other movements.\n7. Inability to pass gas or have a bowel movement.\n\nAppendicitis cannot be cured via medicine alone as the appendix may rupture, leading to peritonitis, a serious inflammation of the abdominal cavity. If diagnosed early, antibiotics can help reduce inflammation and prevent rupture, but surgery is still required to remove the infected appendix.\n\nThe surgical procedure for treating appendicitis is called an appendectomy. It involves making a small incision in the abdomen and removing the appendix. The Merck Manual suggests that open appendectomy or laparoscopic appendectomy 

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [25]:
user_input_13 = system_prompt+"\n"+ "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(user_input_13, max_tokens=500, temperature=0.5, top_k=2)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     613.46 ms
llama_print_timings:      sample time =     294.80 ms /   500 runs   (    0.59 ms per token,  1696.05 tokens per second)
llama_print_timings: prompt eval time =     196.01 ms /    34 tokens (    5.77 ms per token,   173.46 tokens per second)
llama_print_timings:        eval time =   17136.01 ms /   499 runs   (   34.34 ms per token,    29.12 tokens per second)
llama_print_timings:       total time =   18103.06 ms /   533 tokens


"\n\nAccording to the American Academy of Dermatology (AAD), sudden patchy hair loss, also known as alopecia areata, is an autoimmune disease that attacks hair follicles. The exact cause is unknown, but it may be triggered by stress, genetics, or other factors.\n\nDiagnostic Assistance:\nThe diagnosis of alopecia areata is typically made based on the appearance of the bald spots and ruling out other causes of hair loss such as thyroid disorders, nutritional deficiencies, or medications. A dermatologist may perform a scalp biopsy to confirm the diagnosis.\n\nDrug Information:\nThere are several treatments for alopecia areata, including:\n1. Corticosteroids: These drugs can be applied topically or taken orally to reduce inflammation and suppress the immune system's attack on hair follicles.\n2. Immunomodulators: Drugs like minoxidil and anthralin can help stimulate hair growth by promoting blood flow to the affected areas and reducing inflammation.\n3. JAK inhibitors: Newer treatments, s

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [26]:
user_input_14 = system_prompt+"\n"+ "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_input_14, max_tokens=500, temperature=0.5, top_k=2)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     613.46 ms
llama_print_timings:      sample time =     196.52 ms /   348 runs   (    0.56 ms per token,  1770.85 tokens per second)
llama_print_timings: prompt eval time =     216.97 ms /    28 tokens (    7.75 ms per token,   129.05 tokens per second)
llama_print_timings:        eval time =   11884.34 ms /   347 runs   (   34.25 ms per token,    29.20 tokens per second)
llama_print_timings:       total time =   12540.45 ms /   375 tokens


"\n\nAccording to the American Association of Neurological Surgeons (AANS), treatment for a brain injury depends on the severity and location of the injury. Here are some common treatments:\n\n1. Diagnostic Assistance: Imaging tests such as CT scans or MRIs can help diagnose the extent and location of the brain injury.\n2. Drug Information: Depending on the symptoms, medications may be prescribed to manage conditions like seizures, pain, or swelling in the brain. For example, corticosteroids may be used to reduce inflammation, while anticonvulsants can help prevent seizures.\n3. Treatment Plans: Rehabilitation is a key component of treatment for brain injuries. This may include physical therapy to improve motor skills, occupational therapy to help with daily living activities, speech therapy to address communication issues, and cognitive rehabilitation to improve memory and problem-solving abilities.\n4. Specialty Knowledge: Depending on the specific injury and its location, specialize

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [29]:
user_input_15 = system_prompt+"\n"+ "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(user_input_15, max_tokens=500, temperature=0.5, top_k=2)

Llama.generate: prefix-match hit

llama_print_timings:        load time =     613.46 ms
llama_print_timings:      sample time =     276.28 ms /   500 runs   (    0.55 ms per token,  1809.73 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     0 tokens (    -nan ms per token,     -nan tokens per second)
llama_print_timings:        eval time =   17228.13 ms /   500 runs   (   34.46 ms per token,    29.02 tokens per second)
llama_print_timings:       total time =   17916.22 ms /   500 tokens


"\n\nAccording to the American Academy of Orthopaedic Surgeons (AAOS) and the National Library of Medicine's MedlinePlus, here is the necessary information for someone who has fractured their leg during a hiking trip:\n\n1. Necessary Precautions:\n   a. Immobilize the leg: Use a splint or a cast to keep the bone in place and prevent further damage.\n   b. Apply ice: Apply an ice pack for 15-20 minutes at a time, several times a day, to reduce swelling and pain.\n   c. Elevate the leg: Keep the leg elevated above heart level as much as possible to help reduce swelling and pain.\n   d. Avoid weight bearing: Do not put any weight on the injured leg until it has healed. Use crutches or a walker for mobility.\n   e. Seek medical attention: If the fracture is severe, seek immediate medical attention.\n\n2. Treatment Steps:\n   a. Diagnostic Assistance: A healthcare professional will perform an examination and may order imaging tests such as X-rays to confirm the diagnosis of a leg fracture.\

#### Observations:

*   The data provided by the model are more accurate when used with the combination of system and user prompts



## Data Preparation for RAG

### Loading the Data

In [30]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [31]:
manual_pdf_path = "/content/drive/MyDrive/medical_diagnosis_manual.pdf"

In [32]:
pdf_loader = PyMuPDFLoader(manual_pdf_path)

In [33]:
medical_diagnosis_manual = pdf_loader.load()



*   The medical diagnosis manual data has been loaded



### Data Overview

#### Checking the first 5 pages

In [35]:
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(medical_diagnosis_manual[i].page_content,end="\n")

Page Number : 1
ronesh.bhandari@gmail.com
RHNESK7BGA
nt for personal use by ronesh.bhandari@
shing the contents in part or full is liable
Page Number : 2
ronesh.bhandari@gmail.com
RHNESK7BGA
This file is meant for personal use by ronesh.bhandari@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ................................................................................................................................................................................

#### Checking the number of pages

In [36]:
len(medical_diagnosis_manual)

4114

#### Observations:

*   The medical diagnosis manual is of 4114 pages



### Data Chunking

In [51]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap= 40
)

In [52]:
document_chunks = pdf_loader.load_and_split(text_splitter)

In [53]:
len(document_chunks)

8618



*   The length of the document chunks is 8618



In [54]:
document_chunks[0].page_content

'ronesh.bhandari@gmail.com\nRHNESK7BGA\nnt for personal use by ronesh.bhandari@\nshing the contents in part or full is liable'

In [55]:
document_chunks[1].page_content

'ronesh.bhandari@gmail.com\nRHNESK7BGA\nThis file is meant for personal use by ronesh.bhandari@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [56]:
document_chunks[2].page_content

'Table of Contents\n1\nFront    ................................................................................................................................................................................................................\n1\nCover    .......................................................................................................................................................................................................\n2\nFront Matter    ...........................................................................................................................................................................................\n53\n1 - Nutritional Disorders    ...............................................................................................................................................................\n53\nChapter 1. Nutrition: General Considerations    ...........................................................................................

In [57]:
document_chunks[3].page_content

"275\nChapter 23. Approach to the Patient With Liver Disease    ...........................................................................................\n294\nChapter 24. Testing for Hepatic & Biliary Disorders    ......................................................................................................\n305\nChapter 25. Drugs & the Liver    ................................................................................................................................................\n308\nChapter 26. Alcoholic Liver Disease    ....................................................................................................................................\n314\nChapter 27. Fibrosis & Cirrhosis    ............................................................................................................................................\n322\nChapter 28. Hepatitis    ........................................................................................................

In [58]:
document_chunks[-2].page_content

'Y\nYaws 1266-1267\nforest 1379\nY chromosome 3373 (see also Genetic)\nabnormalities of 3005\nYeast infection (see also Fungal infection)\nvaginal 2542, 2544, 2545\nYellow fever 1400, 1429, 1437\nhepatic inflammation in 248\nvaccine against 1172, 1437, 3441\nYellow nail syndrome 732, 1995\npleural effusion in 1997\nYellow skin (see Jaundice)\nYersinia infection 1167, 1256-1257\nY. enterocolitica infection 147\nY. pestis infection 1924\nYew poisoning 3338\nYips 1762\nYo, antibodies to 1056\nYolk sac tumor 2476\nThe Merck Manual of Diagnosis & Therapy, 19th Edition\nY\n4103\nronesh.bhandari@gmail.com\nRHNESK7BGA\nThis file is meant for personal use by ronesh.bhandari@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.'

In [64]:
document_chunks[-1].page_content

"Z\nZafirlukast 1879\nZalcitabine 1451\nin children 2854\nZaleplon 1709\nZanamivir 1407\nin influenza 1407, 1929\nZAP-70 (zeta-associated protein 70) deficiency 1092, 1108\nZavanelli maneuver 2680\nZellweger syndrome 2383, 3023\nZenker's diverticulum 125\nZidovudine 1451, 1453\nin children 2854\nZileuton 1881\nin asthma 1880\nZinc 49, 55, 3431-3432\nin common cold 1405\ndeficiency of 11, 49, 55\nin dermatophytoses 705\npoisoning with 3328, 3353\nrecommended dietary allowances for 50\nreference values for 3499\ntoxicity of 49, 55\ncopper deficiency and 49\nin Wilson's disease 52\nZinc oxide 2233\ngelatin formulation of 646, 672\nZinc pyrithione 647\nZinc shakes 55\nZipper injury 3239, 3240\nZiprasidone\nin agitation 1492\nin bipolar disorder 3059\npoisoning with 3347\nin schizophrenia 1566\nZoledronate 359, 361, 848\nZollinger-Ellison syndrome 95, 199, 200-201, 910\nmastocytosis vs 1125\nMenetrier's disease vs 132\npeptic ulcer disease vs 134\nZolmitriptan 1721\nZolpidem 1709, 3103\nZon

In [62]:
document_chunks[43].page_content

'Professor J. Wollensak via ONJOPH (Plates 8, 9, 15, 16, 17 [top], and 18); Dr. Brooks McCuen via\nONJOPH (Plate 11 [top]); Jonathan J. Dutton via ONJOPH (Plate 12); University of California at San\nDiego via ONJOPH (Plate 13); Professor H.J. Meyer via ONJOPH (Plate 14); Dr. James Garrity (Plate\n19); World Health Organization via ONJOPH (Plate 20); C. Newman (Plate 21) and B. Biller (Plate 22)\nfrom Atlas of Clinical Endocrinology: Neuroendocrinology and Pituitary Disease ; Thomas Habif, MD\n(Plates 24, 26-30, 32-37, 38 [top], 39-47, 49, 51-55, and 60); Prof. Dr. K.W. Ruprecht and Dr. Med. B.\nKaesmann-Kellner via ONJOPH (Plate 25 [top]); Allen W. Mathies, MD, via the Public Health Image\nLibrary of the Centers for Disease Control and Prevention (CDC) (Plate 31); Dennis D. Juranek, MD, via\nthe Public Health Image Library of the CDC (Plate 38 [bottom]) and via www.doctorfungus.com (Plates\n48 and 50); S. Deitcher from Atlas of Clinical Hematology  (Plate 56); E. Joe and N. Soter from 

In [60]:
document_chunks[44].page_content

"Infections (Plate 68); R. Kaufman and D. Brown from Atlas of Clinical Gynecology: Gynecologic\nPathology (Plate 69); J.D. Sobel from Atlas of Infectious Diseases: Fungal Infections  (Plate 71); A.\nOster and R. Rosa from Atlas of Ophthalmology  (Plate 72); I. Scott, R. Warman, and T. Murray from\nAtlas of Ophthalmology  (Plate 73); J. Burns and M. Glode from Atlas of Infectious Diseases: Pediatric\nInfectious Diseases (Plate 74); Steven E. Wolf, MD (Plates 75 and 76).\nContributors\nBOLA ADAMOLEKUN, MD\nDirector of Epilepsy, Department of Neurology,\nUniversity of Tennessee Health Science Center\nSeizure Disorders\nNEIL B. ALEXANDER, MD\nProfessor, Department of Internal Medicine,\nDivision of Geriatric Medicine, University of\nMichigan; Associate Professor for Research;\nDirector, VA Ann Arbor Health Care System,\nGeriatrics Research, Education and Clinical\nCenter\nFalls in the Elderly\nROY D. ALTMAN, MD\nProfessor of Medicine, Division of\nRheumatology and Immunology, University\no

### Embedding

### Vector Database

### Retriever

### System and User Prompt Template

### Response Function

In [ ]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

### Fine-tuning

## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [ ]:
groundedness_rater_system_message  = ""

In [ ]:
relevance_rater_system_message = ""

In [ ]:
user_message_template = ""

In [ ]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

## Actionable Insights and Business Recommendations

<font size=6 color='blue'>Power Ahead</font>
___